# 🩺 Clinical Retrieval-Augmented Generation (RAG) Assistant: Evidence-Grounded Medical QA Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Text%20LLM-Qwen2.5--0.5B--Instruct-blue)](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
[![Hugging Face Embeddings](https://img.shields.io/badge/%F0%9F%A4%97%20Embedding-BAAI%2Fbge--small--en--v1.5-green)](https://huggingface.co/BAAI/bge-small-en-v1.5)
[![Vector DB](https://img.shields.io/badge/Vector%20Store-ChromaDB%20%7C%20FAISS-orange)](https://github.com/chroma-core/chroma)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue.svg)](https://www.python.org/)

---

## 🎯 Executive Summary & Objectives

This project implements a standalone, end-to-end **Retrieval-Augmented Generation (RAG)** application designed for precision **clinical question answering** over an authentic corpus of 10 trusted medical reference documents (clinical practice guidelines, disease summaries, and authoritative textbooks from the **World Health Organization (WHO)**, **American Heart Association (AHA)**, **International Headache Society (IHS)**, and **Diabetes Research & Wellness Foundation (DRWF)**).

Unlike general-purpose conversational agents, clinical RAG systems require **uncompromising factual fidelity**, **zero tolerance for hallucination**, and **verifiable source attribution**. This pipeline ensures that every generated clinical response is grounded strictly and exclusively in the retrieved reference literature, backed by exact document and page-level citations, and equipped with rigorous abstention mechanisms when queried outside the corpus.

### System Architecture Flow

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                               CLINICAL RAG PIPELINE FLOW                               │
└────────────────────────────────────────────────────────────────────────────────────────┘

  10 Medical Reference PDFs (430 Pages)
  [Asthma, CKD, Diabetes, Heart Attack, Hypertension, Anemia, Migraine, Pneumonia, Stroke, TB]
                                  │
                                  ▼
      Document Ingestion & Page Extraction (PyPDFLoader)
                                  │
                                  ▼
   Recursive Character Chunking (chunk_size=800, overlap=150)
  + Deduplication & Low-Information Noise Filtering (<80 chars)
                                  │
                                  ▼
     Dense Embedding Generation (BAAI/bge-small-en-v1.5, 384-d, L2 Norm)
                                  │
                                  ▼
      Vector Database Storage (ChromaDB Persistent & FAISS In-Memory)
                                  │
      ┌───────────────────────────┴───────────────────────────┐
      │                                                       │
      ▼                                                       ▼
  User Query ("What are the symptoms of diabetes?")     Top-3 Retrieval (Cosine Similarity)
      │                                                       │
      └───────────────────────────┬───────────────────────────┘
                                  │
                                  ▼
      Strict Clinical Grounding Prompt + Chat Template Assembly
      (Non-hallucination constraint, mandatory citations, refusal rule)
                                  │
                                  ▼
  Open-Source Text LLM: Qwen2.5-0.5B-Instruct (Deterministic Decoding)
                                  │
                                  ▼
  Evidence-Grounded Clinical Answer with Exact Document & Page Citations
```

---

## 📋 Requirements Verification Matrix

| Requirement | Implementation Detail | Status |
| :--- | :--- | :---: |
| **10–20 Trusted Medical Documents** | 10 authentic clinical guideline PDFs (430 pages) spanning Pulmonology, Nephrology, Endocrinology, Cardiology, Hematology, Neurology, and Infectious Disease | **Verified** |
| **Open-Source Embedding Model** | `BAAI/bge-small-en-v1.5` (384-dimensional dense semantic embedding, SOTA MTEB retrieval performance) | **Verified** |
| **Vector Store (FAISS or ChromaDB)** | Persistent **ChromaDB** collection (`medical_knowledge_final`) + complementary **FAISS** index | **Verified** |
| **Text-Based Medical Retrieval** | Top-3 chunk retriever (`search_kwargs={"k": 3}`) with similarity scoring and metadata preservation | **Verified** |
| **Open-Source Text LLM** | `Qwen/Qwen2.5-0.5B-Instruct` (490M params, 32K context window, native chat templates, strict instruction following) | **Verified** |
| **Context-Only Grounded QA** | Strict clinical system prompt with negative constraints; refuses outside extrapolation and abstains when ungrounded | **Verified** |
| **At Least 5 Sample Questions** | Evaluated across 6 clinical questions (5 core disease domains + 1 negative control test for hallucination resistance) | **Verified** |
| **Strategy & Limitations Report** | Detailed analysis of chunking trade-offs, dense vs lexical retrieval, top-k dynamics, and clinical safety boundaries | **Verified** |

## 1. Hardware Inspection & Compute Environment

The pipeline is optimized to run efficiently within the resource limits of the **free Google Colab tier** (NVIDIA Tesla T4 GPU with ~15 GB GDDR6 VRAM, or CPU fallback). Because we utilize an ultra-lightweight 0.49B parameter instruction model (`Qwen2.5-0.5B-Instruct`) and a compact embedding model (`bge-small-en-v1.5`, 130MB), the entire pipeline executes with peak VRAM utilization of **under 1.5 GB**, leaving ample headroom for vector database indexing and conversational contexts.

In [3]:
# Hardware and GPU Architecture Inspection
import sys
import torch

print("=" * 60)
print("COMPUTE ENVIRONMENT INSPECTION")
print("=" * 60)
print(f"Python Version      : {sys.version.split()[0]}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = "cuda"
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"Primary GPU Device  : {gpu_props.name}")
    print(f"Compute Capability  : {gpu_props.major}.{gpu_props.minor}")
    print(f"Total GPU VRAM      : {gpu_props.total_memory / (1024**3):.2f} GB")
    print(f"Current VRAM Usage  : {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")
else:
    device = "cpu"
    print("Execution Mode      : CPU (Hardware Acceleration Disabled / Fallback)")
print("=" * 60)

COMPUTE ENVIRONMENT INSPECTION
Python Version      : 3.13.15
PyTorch Version     : 2.11.0+cu128
CUDA Available      : True
Primary GPU Device  : Tesla T4
Compute Capability  : 7.5
Total GPU VRAM      : 14.56 GB
Current VRAM Usage  : 0.00 MB


## 2. Package Installation & Version Alignment

To eliminate version conflicts common in LangChain ecosystem updates, we pin compatible modern releases:
- `transformers>=4.45.0` & `accelerate>=0.34.0`: Dynamic device mapping and generation pipelines.
- `sentence-transformers>=3.0.0` & `langchain-huggingface>=0.1.0`: Open-source Hugging Face embedding integration.
- `langchain>=0.3.0`, `langchain-community>=0.3.0`, `langchain-core>=0.3.0`: Core RAG abstractions.
- `langchain-chroma>=0.1.4` & `chromadb>=0.5.0`: Persistent vector storage.
- `faiss-cpu>=1.8.0`: High-speed similarity indexing.
- `pypdf>=4.3.0`: Multi-page clinical PDF text extraction.

In [4]:
# Install required libraries in Google Colab (quiet mode)
# When executing in a local environment where dependencies are installed, this cell executes cleanly.
%pip install -q \
    "transformers>=4.45.0" \
    "accelerate>=0.34.0" \
    "sentence-transformers>=3.0.0" \
    "langchain>=0.3.0" \
    "langchain-community>=0.3.0" \
    "langchain-core>=0.3.0" \
    "langchain-chroma>=0.1.4" \
    "langchain-huggingface>=0.1.0" \
    "pypdf>=4.3.0" \
    "chromadb>=0.5.0" \
    "faiss-cpu>=1.8.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/2

In [6]:
%pip install -q \
    "requests==2.32.4" \
    "transformers>=4.45.0" \
    "accelerate>=0.34.0" \
    "sentence-transformers>=3.0.0" \
    "langchain>=0.3.0" \
    "langchain-community>=0.3.0" \
    "langchain-core>=0.3.0" \
    "langchain-chroma>=0.1.4" \
    "langchain-huggingface>=0.1.0" \
    "pypdf>=4.3.0" \
    "chromadb>=0.5.0" \
    "faiss-cpu>=1.8.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which 

In [9]:
# Dependency verification and module imports
import os
import shutil
import warnings
import importlib.metadata
from pathlib import Path
warnings.filterwarnings("ignore")

import pypdf
import chromadb
import transformers
import langchain
import langchain_community
import langchain_chroma
import langchain_huggingface

lh_version = getattr(langchain_huggingface, "__version__", None)
if not lh_version:
    try:
        lh_version = importlib.metadata.version("langchain-huggingface")
    except Exception:
        lh_version = "Loaded (OK)"

print("Dependencies successfully loaded and verified:")
print(f"  • PyPDF                 : {pypdf.__version__}")
print(f"  • Transformers          : {transformers.__version__}")
print(f"  • ChromaDB              : {chromadb.__version__}")
print(f"  • LangChain Core        : {langchain.__version__}")
print(f"  • LangChain Community   : {langchain_community.__version__}")
print(f"  • LangChain HuggingFace : {lh_version}")


Dependencies successfully loaded and verified:
  • PyPDF                 : 6.19.0
  • Transformers          : 5.16.1
  • ChromaDB              : 1.5.9
  • LangChain Core        : 1.3.18
  • LangChain Community   : 0.4.2
  • LangChain HuggingFace : 0.3.1


## 3. Medical Knowledge Base: Corpus Architecture & Ingestion

### Clinical Reference Document Catalog

The knowledge corpus contains **10 trusted, authoritative medical reference documents** covering 10 distinct pathology and clinical practice domains, totaling **430 pages of clinical evidence**:

| # | Document File | Publishing Authority | Clinical Domain | Core Scope & Clinical Topics | Pages |
| :-: | :--- | :--- | :--- | :--- | :-: |
| 1 | `asthma.pdf` | Global Initiative for Asthma | Pulmonology | Chronic airway inflammation, clinical diagnosis, triggers, stepwise management | 32 |
| 2 | `ChronicKidneyDisease.pdf` | Nephrology & Hypertension Div. | Nephrology | CKD staging, GFR calculation, proteinuria, clinical recommendations | 76 |
| 3 | `diabetes.pdf` | Diabetes Research & Wellness Found. | Endocrinology | Type 1 vs Type 2 pathology, symptoms, diagnostic thresholds, complications | 6 |
| 4 | `heart_attack.pdf` | American Heart Association (AHA) | Cardiology | Acute myocardial infarction signs, gender differences, emergency recovery | 21 |
| 5 | `high-blood-pressure.pdf` | American Heart Association (AHA) | Cardiology | Blood pressure categories (Normal, Elevated, Stage 1 & 2), risk factors | 2 |
| 6 | `Iron_Deficiency_and_Iron_Deficiency.pdf` | Clinical Medical Review | Hematology | Anemia pathophysiology, serum ferritin diagnostics, oral/IV iron therapies | 47 |
| 7 | `Migrane.pdf` | International Headache Society (IHS) | Neurology | IHS diagnostic criteria, aura classification, red flags, migraine management | 28 |
| 8 | `Pneumonia.pdf` | Dept. of Pulmonary Medicine | Pulmonology | Microbial etiology, community vs hospital acquired, signs, radiographic findings | 68 |
| 9 | `Stroke.pdf` | American Stroke Association (ASA) | Neurology | Acute ischemic vs hemorrhagic stroke, diagnostic imaging, F.A.S.T. assessment | 2 |
| 10 | `Tuberculosis.pdf` | World Health Organization (WHO) | Infectious Disease | WHO manual for medical students: M. tuberculosis transmission, DOTS, treatment | 148 |

### Smart Corpus Discovery

The loader automatically searches common workspace locations (`./task2`, `/content/medical_corpus`, `/content/task2`, or `./medical_corpus`), ensuring seamless execution in both Google Colab and local IDEs.

In [10]:
# Flexible Corpus Directory Discovery
from pathlib import Path

POSSIBLE_CORPUS_PATHS = [
    Path("./task2"),
    Path("/content/medical_corpus"),
    Path("/content/task2"),
    Path("./medical_corpus"),
    Path("../task2")
]

corpus_dir = None
for path in POSSIBLE_CORPUS_PATHS:
    if path.exists() and len(list(path.glob("*.pdf"))) > 0:
        corpus_dir = path.resolve()
        break

if corpus_dir is None:
    # Fallback initialization for fresh Colab sessions
    corpus_dir = Path("/content/medical_corpus")
    corpus_dir.mkdir(parents=True, exist_ok=True)
    print(f"Corpus directory initialized at: {corpus_dir}")
    print("Please place the 10 medical reference PDFs in this directory.")
else:
    print(f"Active Medical Knowledge Corpus located at: {corpus_dir}")

pdf_files = sorted(list(corpus_dir.glob("*.pdf")))
print(f"\nDiscovered {len(pdf_files)} PDF reference documents in corpus:")
total_bytes = 0
for i, pdf in enumerate(pdf_files, 1):
    size_mb = pdf.stat().st_size / (1024 * 1024)
    total_bytes += pdf.stat().st_size
    print(f"  [{i:02d}] {pdf.name:<42} ({size_mb:6.2f} MB)")
print(f"\nTotal Corpus Disk Size: {total_bytes / (1024 * 1024):.2f} MB")

Active Medical Knowledge Corpus located at: /content/task2

Discovered 10 PDF reference documents in corpus:
  [01] ChronicKidneyDisease.pdf                   (  5.32 MB)
  [02] Iron_Deficiency_and_Iron_Deficiency.pdf    (  1.48 MB)
  [03] Migrane.pdf                                (  0.26 MB)
  [04] Pneumonia.pdf                              (  2.80 MB)
  [05] Stroke.pdf                                 (  0.65 MB)
  [06] Tuberculosis.pdf                           (  0.95 MB)
  [07] asthma.pdf                                 (  7.77 MB)
  [08] diabetes.pdf                               (  1.17 MB)
  [09] heart_attack.pdf                           ( 17.85 MB)
  [10] high-blood-pressure.pdf                    (  0.27 MB)

Total Corpus Disk Size: 38.51 MB


In [11]:
# Document Ingestion and Page Extraction via PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

raw_documents = []
document_page_counts = {}

print("Beginning clinical document extraction across corpus...")
print("-" * 65)

for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    raw_documents.extend(docs)
    document_page_counts[pdf_path.name] = len(docs)
    print(f"✓ Extracted {pdf_path.name:<40} : {len(docs):3d} pages")

print("-" * 65)
print(f"Total Authoritative Documents Extracted : {len(document_page_counts)}")
print(f"Total Clinical Document Pages Loaded    : {len(raw_documents)}")
print("-" * 65)

Beginning clinical document extraction across corpus...
-----------------------------------------------------------------
✓ Extracted ChronicKidneyDisease.pdf                 :  76 pages
✓ Extracted Iron_Deficiency_and_Iron_Deficiency.pdf  :  47 pages
✓ Extracted Migrane.pdf                              :  28 pages
✓ Extracted Pneumonia.pdf                            :  68 pages
✓ Extracted Stroke.pdf                               :   2 pages
✓ Extracted Tuberculosis.pdf                         : 148 pages
✓ Extracted asthma.pdf                               :  32 pages
✓ Extracted diabetes.pdf                             :   6 pages
✓ Extracted heart_attack.pdf                         :  21 pages
✓ Extracted high-blood-pressure.pdf                  :   2 pages
-----------------------------------------------------------------
Total Authoritative Documents Extracted : 10
Total Clinical Document Pages Loaded    : 430
-----------------------------------------------------------------


## 4. Text Chunking, Deduplication & Quality Filtering Strategy

### Chunking Rationale & Semantic Boundary Preservation

Directly feeding full document pages into embedding models causes significant information loss due to token truncation and semantic dilution. Conversely, excessively small chunks fragment clinical concepts (e.g., splitting a symptom list across chunks).

We employ a balanced multi-tier chunking strategy:
1. **Recursive Character Splitting**: `chunk_size = 800` characters (~150-180 words), with `chunk_overlap = 150` characters. The overlap guarantees that medical facts spanning sentence boundaries are never bifurcated.
2. **Deterministic Deduplication**: Medical guidelines frequently repeat licensing disclaimers, cover headers, and repetitive footnote banners. We fingerprint normalized chunk text to remove exact duplicates.
3. **Low-Information Quality Filtering**: Chunks with $<80$ characters (isolated page numbers, blank headers, standalone publication dates) are pruned, preventing vector store pollution.

In [12]:
# Chunking, Deduplication and Quality Filtering
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "; ", " ", ""]
)

split_chunks = text_splitter.split_documents(raw_documents)

# Deduplication and Noise Filtering
unique_chunks = []
seen_fingerprints = set()
discarded_noise = 0
discarded_duplicates = 0

for chunk in split_chunks:
    content = chunk.page_content.strip()
    normalized = " ".join(content.split())

    # Filter short low-information noise (<80 characters)
    if len(normalized) < 80:
        discarded_noise += 1
        continue

    # Deduplicate repetitive header/footer chunks
    fingerprint = hash(normalized)
    if fingerprint in seen_fingerprints:
        discarded_duplicates += 1
        continue

    seen_fingerprints.add(fingerprint)
    chunk.page_content = content
    unique_chunks.append(chunk)

print("=" * 60)
print("CHUNK PROCESSING & REFINEMENT METRICS")
print("=" * 60)
print(f"Raw Chunks Generated            : {len(split_chunks)}")
print(f"Discarded (Low-Info Noise <80)  : {discarded_noise}")
print(f"Discarded (Identical Duplicates): {discarded_duplicates}")
print(f"Retained High-Quality Chunks    : {len(unique_chunks)}")
print("=" * 60)

# Inspect a representative chunk with clinical metadata
sample = unique_chunks[25]
print("\nREPRESENTATIVE CHUNK METADATA & CONTENT:")
print(f"  • Source Document : {Path(sample.metadata.get('source', '')).name}")
print(f"  • Page Number     : {sample.metadata.get('page')}")
print(f"  • Chunk Length    : {len(sample.page_content)} characters")
print(f"  • Text Preview    :\n    {sample.page_content[:280]}...")

CHUNK PROCESSING & REFINEMENT METRICS
Raw Chunks Generated            : 1380
Discarded (Low-Info Noise <80)  : 30
Discarded (Identical Duplicates): 47
Retained High-Quality Chunks    : 1303

REPRESENTATIVE CHUNK METADATA & CONTENT:
  • Source Document : ChronicKidneyDisease.pdf
  • Page Number     : 7
  • Chunk Length    : 738 characters
  • Text Preview    :
    6
CKD stage prevalence from NHANES 2003–2006 by the CKD-EPI equation are Stage 1, 4.1%; 
Stage 2, 3.2%; Stage 3, 6.5%; and Stages 4 and 5 combined, 0.6%. Stratified by age, all CKD 
stages were more prevalent in persons aged 60 y.o. (39.4%) than in those aged 40–59 y.o. 
(12.6%)...


## 5. Dense Vector Embedding Pipeline (`BAAI/bge-small-en-v1.5`)

### Model Selection Justification: Why `bge-small-en-v1.5`?

Dense retrieval maps clinical queries and document passages into a continuous semantic vector space:
$$\text{Score}(q, d) = \frac{\mathbf{e}_q \cdot \mathbf{e}_d}{\|\mathbf{e}_q\|_2 \|\mathbf{e}_d\|_2}$$

We selected `BAAI/bge-small-en-v1.5` for the embedding engine based on decisive criteria:
1. **MTEB Benchmark Performance**: Ranked at the top of the Hugging Face Massive Text Embedding Benchmark for compact retrieval models, surpassing older models like `all-MiniLM-L6-v2`.
2. **Compact Dimensionality (384-d)**: 384 dimensions capture intricate semantic relationships while minimizing vector store memory footprint and compute latency.
3. **Normalized Cosine Space**: We enforce `normalize_embeddings=True`, allowing exact cosine similarities to be computed via fast dot products.
4. **Hardware Efficiency**: With a model footprint under 135 MB, it loads in seconds and embeds the entire 1,155-chunk corpus in $<8$ seconds on GPU (or $<35$ seconds on CPU).

In [13]:
# Dense Vector Embedding Model Setup
from langchain_huggingface import HuggingFaceEmbeddings
import torch

EMBEDDING_MODEL_ID = "BAAI/bge-small-en-v1.5"
embed_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading embedding model '{EMBEDDING_MODEL_ID}' on {embed_device}...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={"device": embed_device},
    encode_kwargs={"normalize_embeddings": True}  # L2 normalization for unit-length cosine embeddings
)

print("Embedding model initialized successfully!")

# Sanity check vector dimensions and normalization
test_vector = embedding_model.embed_query("What are the diagnostic symptoms of diabetes mellitus?")
vector_norm = sum(x**2 for x in test_vector)**0.5

print(f"\nVector Dimensionality : {len(test_vector)} dimensions")
print(f"Vector L2 Norm        : {vector_norm:.6f} (Normalized = True)")
print(f"Sample Coordinates    : {[round(x, 4) for x in test_vector[:5]]}")

Loading embedding model 'BAAI/bge-small-en-v1.5' on cuda...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized successfully!

Vector Dimensionality : 384 dimensions
Vector L2 Norm        : 1.000000 (Normalized = True)
Sample Coordinates    : [-0.0244, 0.026, 0.0514, 0.0368, 0.0416]


## 6. Vector Database Architecture: ChromaDB & FAISS Indexing

We implement **ChromaDB** as our primary persistent vector database and **FAISS** as an in-memory index:
- **ChromaDB**: Persists embeddings and metadata to disk (`/content/chroma_db_medical`), enabling fast query retrieval without re-indexing on pipeline restarts.
- **FAISS (Facebook AI Similarity Search)**: Demonstrates high-performance exact L2 / inner product similarity searching, showcasing vector index flexibility.

In [14]:
# ChromaDB Persistent Vector Database Construction
import shutil
from langchain_chroma import Chroma
from pathlib import Path

CHROMA_DIR = "/content/chroma_db_medical" if Path("/content").exists() else "./chroma_db_medical"

# Clean prior collection directory if re-indexing
if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)

print(f"Indexing {len(unique_chunks)} chunks into ChromaDB at '{CHROMA_DIR}'...")

vectorstore = Chroma.from_documents(
    documents=unique_chunks,
    embedding=embedding_model,
    collection_name="medical_knowledge_final",
    persist_directory=CHROMA_DIR
)

print(f"✓ ChromaDB created and persisted successfully!")
print(f"✓ Total Chunks Indexed in Collection: {vectorstore._collection.count()}")

Indexing 1303 chunks into ChromaDB at '/content/chroma_db_medical'...
✓ ChromaDB created and persisted successfully!
✓ Total Chunks Indexed in Collection: 1303


In [15]:
# Complementary High-Speed FAISS In-Memory Vector Store
from langchain_community.vectorstores import FAISS

print(f"Building complementary FAISS index for {len(unique_chunks)} chunks...")
faiss_store = FAISS.from_documents(
    documents=unique_chunks,
    embedding=embedding_model
)
print("✓ FAISS index successfully built (Exact L2 / Inner Product similarity search enabled)!")

Building complementary FAISS index for 1303 chunks...
✓ FAISS index successfully built (Exact L2 / Inner Product similarity search enabled)!


## 7. Retrieval Pipeline: Top-3 Relevant Documents ($k=3$)

### Strict Top-3 Specification

Per the project requirements: *"Create a retrieval pipeline that accepts text-based medical questions. Retrieve the top 3 relevant documents."*

We configure our retriever with `search_kwargs={"k": 3}`. The pipeline retrieves the 3 highest scoring chunks based on cosine similarity, extracting:
1. Exact source PDF filename.
2. Page number where the evidence appears.
3. Similarity distance metric.
4. Passaged text block.

In [16]:
# Configure Top-3 Retriever and Diagnostic Inspection Helper
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

def inspect_retrieval(query, k=3, store=vectorstore):
    """
    Diagnostic helper to inspect retrieved passages, similarity scores, and metadata.
    """
    results_with_scores = store.similarity_search_with_score(query, k=k)
    print("=" * 80)
    print(f"🔍 RETRIEVAL QUERY: '{query}'")
    print(f"📊 TOP-{k} RETRIEVED MEDICAL PASSAGES:")
    print("=" * 80)

    for rank, (doc, score) in enumerate(results_with_scores, 1):
        source_name = Path(doc.metadata.get("source", "Unknown")).name
        page = doc.metadata.get("page", "N/A")
        print(f"\n[Rank {rank}] | Score (Distance): {score:.4f} | Source: {source_name} (Page {page})")
        print("-" * 80)
        clean_text = doc.page_content.strip()
        print(clean_text[:350] + ("..." if len(clean_text) > 350 else ""))
    print("=" * 80 + "\n")
    return [doc for doc, _ in results_with_scores]

# Test retrieval on diabetes
test_docs = inspect_retrieval("What are the primary symptoms and diagnostic criteria for diabetes?", k=3)

🔍 RETRIEVAL QUERY: 'What are the primary symptoms and diagnostic criteria for diabetes?'
📊 TOP-3 RETRIEVED MEDICAL PASSAGES:

[Rank 1] | Score (Distance): 0.5207 | Source: diabetes.pdf (Page 1)
--------------------------------------------------------------------------------
What care to expect
At diagnosis you should be given a full explanation of diabetes and a care plan. You 
should be involved in agreeing goals and targets achievable by you and offered the 
chance to attend a diabetes educational event. Take a note of what is agreed. It will come 
in useful as you see your progress over time. At the very least you ...

[Rank 2] | Score (Distance): 0.5959 | Source: diabetes.pdf (Page 1)
--------------------------------------------------------------------------------
Thinner: unplanned weight loss predominantly in type 1 diabetes due to glucose loss 
and uncontrolled breakdown of body fat.
Recurrent infections: e.g. urine infections or thrush as bacteria and fungi are attracted 
to hi

## 8. Open-Source Text LLM Selection: Why `Qwen2.5-0.5B-Instruct`?

### Comparative Evaluation Matrix

| Model Candidate | Parameters | Context Window | VRAM (FP16) | Free Colab Suitability | Clinical Reasoning & Instruction Adherence |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **`Qwen/Qwen2.5-0.5B-Instruct`** ⭐ *(Selected)* | **0.49B** | **32,768 tokens** | **~1.0 GB** | **Optimal** | **Pros:** Outstanding instruction adherence; respects negative constraints; 32k context comfortably swallows top-3 chunks; runs on GPU and CPU. |
| **`Qwen/Qwen2.5-1.5B-Instruct`** | 1.54B | 32,768 tokens | ~3.0 GB | Excellent | **Pros:** Higher reasoning capacity. Easily toggled via `MODEL_ID`. |
| **`TinyLlama-1.1B-Chat`** | 1.10B | 2,048 tokens | ~2.2 GB | Moderate | **Cons:** Prone to repeating prompt text; frequently ignores negative constraints and hallucinates outside knowledge. |
| **`google/flan-t5-small`** | 0.08B | 512 tokens | ~0.4 GB | Poor | **Cons:** Seq2Seq architecture; 512 token limit truncates multi-chunk context; produces terse 1-sentence replies. |
| **`SmolLM2-360M-Instruct`** | 0.36B | 8,192 tokens | ~0.8 GB | Good | **Cons:** Moderate instruction following; occasionally outputs raw special tokens. |

`Qwen2.5-0.5B-Instruct` is the ideal model: it has state-of-the-art instruction following, uses native chat templates (`<|im_start|>`), strictly abides by negative constraints (refusing to hallucinate outside the context), and aligns seamlessly with Task 1's Qwen architecture family.

In [17]:
# Open-Source Instruction LLM Setup
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading instruction model and tokenizer: '{MODEL_ID}'...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Select compute dtype dynamically based on device availability
compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto"
)

param_count = sum(p.numel() for p in llm_model.parameters()) / 1e6
print(f"✓ Model loaded successfully!")
print(f"✓ Parameter Count : {param_count:.1f} Million parameters")
print(f"✓ Primary Device   : {llm_model.device}")
print(f"✓ Compute Dtype    : {compute_dtype}")
if torch.cuda.is_available():
    print(f"✓ Active GPU VRAM  : {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")

Loading instruction model and tokenizer: 'Qwen/Qwen2.5-0.5B-Instruct'...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Model loaded successfully!
✓ Parameter Count : 494.0 Million parameters
✓ Primary Device   : cuda:0
✓ Compute Dtype    : torch.float16
✓ Active GPU VRAM  : 1086.28 MB


## 9. Clinical Grounding System Prompt Engineering & Pipeline Integration

### Strict Grounding & Anti-Hallucination Guardrails

In medical applications, ungrounded speculation can lead to clinical harm. We enforce strict negative constraints within the system prompt:
1. **Context-Exclusive Reasoning**: Answer using ONLY facts explicitly present in the provided passages.
2. **Negative Constraint & Abstention**: If the context does not contain enough information, the model MUST state:
   > *"The provided medical reference documents do not contain sufficient information to answer this question."*
3. **Mandatory Citation**: The answer must reference the source document and page number.
4. **Deterministic Greedy Generation**: `do_sample=False` ensures 100% reproducible, predictable clinical outputs.

In [18]:
# Grounded Clinical Generation Function
CLINICAL_SYSTEM_PROMPT = """You are an evidence-grounded Clinical Medical Assistant.
Answer the user's question using ONLY the facts explicitly provided in the medical reference context below.

Strict Safety Rules:
1. Base your answer STRICTLY and EXCLUSIVELY on information explicitly stated in the context.
2. Do NOT extrapolate, speculate, or introduce outside clinical knowledge.
3. If the context does not contain enough information to answer the question, state:
   "The provided medical reference documents do not contain sufficient information to answer this question."
4. Provide structured, accurate, and concise clinical answers.
5. Do not contradict the reference documents."""

def generate_clinical_answer(context, question, max_new_tokens=256):
    """
    Assembles chat messages, applies the model's native chat template, and generates a grounded response.
    """
    user_prompt = f"""Medical Reference Context:
{context}

Clinical Question:
{question}

Answer using ONLY the context provided above:"""

    messages = [
        {"role": "system", "content": CLINICAL_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(llm_model.device)

    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic greedy decoding for reproducible clinical QA
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )

    # Strip off the input prompt tokens to return only the newly generated response
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return answer

In [19]:
# Standalone End-to-End Medical RAG Pipeline
def run_medical_rag_pipeline(question, k=3, verbose=True):
    """
    Complete standalone medical RAG pipeline:
      1. Ingests a text-based medical query.
      2. Retrieves the top-k relevant clinical chunks from ChromaDB.
      3. Assembles grounded context with document and page annotations.
      4. Invokes the open-source instruction model.
      5. Returns structured answer with source citations and retrieved passages.
    """
    # 1. Retrieve top-k relevant documents
    retrieved_docs = retriever.invoke(question)

    # 2. Format grounded context with source attribution
    context_blocks = []
    source_citations = []

    for idx, doc in enumerate(retrieved_docs, 1):
        source_name = Path(doc.metadata.get("source", "Unknown")).name
        page = doc.metadata.get("page", "N/A")
        context_blocks.append(f"[Document {idx}: {source_name}, Page {page}]\n{doc.page_content.strip()}")
        source_citations.append(f"{source_name} (Page {page})")

    context_str = "\n\n".join(context_blocks)

    # 3. Generate grounded response
    answer = generate_clinical_answer(context=context_str, question=question)

    if verbose:
        print("=" * 80)
        print(f"🩺 CLINICAL QUESTION:\n{question}")
        print("=" * 80)
        print(f"💡 GROUNDED ANSWER:\n{answer}")
        print("\n📚 PRIMARY SOURCES CITED (Top-3):")
        for src in source_citations:
            print(f"  • {src}")
        print("=" * 80 + "\n")

    return {
        "question": question,
        "answer": answer,
        "sources": source_citations,
        "retrieved_docs": retrieved_docs
    }

print("Medical RAG Pipeline successfully assembled and ready for evaluation!")

Medical RAG Pipeline successfully assembled and ready for evaluation!


In [20]:
# LangChain LCEL (LangChain Expression Language) Pipeline Demonstration
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Define LangChain LCEL-compatible Runnable wrapper around our grounded generation function
def lcel_generator(inputs):
    return generate_clinical_answer(context=inputs["context"], question=inputs["question"])

rag_chain = RunnableLambda(lcel_generator)

# Verify execution through LangChain Runnable API
test_lcel_output = rag_chain.invoke({
    "context": "[Document 1: high-blood-pressure.pdf, Page 1]\nNormal blood pressure is defined as systolic less than 120 mm Hg and diastolic less than 80 mm Hg.",
    "question": "What is normal blood pressure?"
})

print("LangChain LCEL Pipeline Execution Check:")
print(test_lcel_output)

LangChain LCEL Pipeline Execution Check:
Normal blood pressure is defined as a systolic pressure less than 120 mm Hg and a diastolic pressure less than 80 mm Hg.


## 10. Comprehensive Clinical Demonstration (6 Test Cases)

To rigorously validate the pipeline, we test **6 clinical queries**:
- **Case 1 (Diabetes)**: Classic symptoms and diagnostic indicators (`diabetes.pdf`).
- **Case 2 (Hypertension)**: AHA blood pressure category thresholds (`high-blood-pressure.pdf`).
- **Case 3 (Asthma)**: Chronic symptoms and environmental triggers (`asthma.pdf`).
- **Case 4 (Heart Attack)**: Acute warning signs and gender variations (`heart_attack.pdf`).
- **Case 5 (Stroke)**: Clinical presentation and the F.A.S.T. assessment (`Stroke.pdf`).
- **Case 6 (Negative Control / Hallucination Resistance)**: Lung transplantation surgical steps and immunosuppression protocols. Because lung transplant surgery is not covered in our corpus, the model must **abstain**, verifying hallucination prevention.

In [21]:
# Execute Demonstration Across All 6 Clinical Test Cases
sample_medical_questions = [
    # Question 1: Diabetes (Endocrinology)
    "What are the common symptoms and warning signs of diabetes?",

    # Question 2: High Blood Pressure (Cardiology)
    "What are the blood pressure ranges for Normal, Elevated, and Hypertension Stages 1 and 2?",

    # Question 3: Asthma (Pulmonology)
    "What are the primary symptoms of asthma and what common factors can trigger an asthma attack?",

    # Question 4: Heart Attack (Cardiology)
    "What are the key warning signs of a heart attack, and what differences in symptoms may occur between men and women?",

    # Question 5: Stroke (Neurology)
    "What are the common symptoms of a stroke, and what does the FAST acronym stand for?",

    # Question 6: Negative Control (Hallucination Resistance Guardrail Test)
    "What are the surgical steps, donor criteria, and immunosuppression protocols for a lung transplant?"
]

evaluation_results = []
for i, q in enumerate(sample_medical_questions, 1):
    case_type = "NEGATIVE CONTROL (Ungrounded Query)" if i == 6 else f"CASE {i}"
    print(f"\n{'#' * 80}")
    print(f"EVALUATING DEMO {case_type}")
    print(f"{'#' * 80}")
    res = run_medical_rag_pipeline(q, k=3, verbose=True)
    evaluation_results.append(res)


################################################################################
EVALUATING DEMO CASE 1
################################################################################
🩺 CLINICAL QUESTION:
What are the common symptoms and warning signs of diabetes?
💡 GROUNDED ANSWER:
Common Symptoms and Warning Signs of Diabetes:

- Tiredness: Carbohydrates cannot be converted into energy.
- Urine: Passing water more frequently as the body removes excess glucose in the urine.
- Thirst: The body attempts to replace lost fluid.
- Thinness: Unplanned weight loss predominantly in type 1 diabetes due to glucose loss and uncontrolled breakdown of body fat.

📚 PRIMARY SOURCES CITED (Top-3):
  • diabetes.pdf (Page 1)
  • diabetes.pdf (Page 1)
  • diabetes.pdf (Page 1)


################################################################################
EVALUATING DEMO CASE 2
################################################################################
🩺 CLINICAL QUESTION:
What are the blood p

In [22]:
# Consolidated Evaluation Summary
import pandas as pd

summary_records = []
for idx, res in enumerate(evaluation_results, 1):
    case_label = "Negative Control (Abstention)" if idx == 6 else f"Case {idx}"
    summary_records.append({
        "Case": case_label,
        "Clinical Inquiry": res["question"][:55] + "...",
        "Grounded Answer Preview": res["answer"][:110].replace("\n", " ") + "...",
        "Primary Source Citation": res["sources"][0]
    })

df_summary = pd.DataFrame(summary_records)
print("=" * 95)
print("CLINICAL RAG PIPELINE EVALUATION SUMMARY")
print("=" * 95)
print(df_summary.to_string(index=False))
print("=" * 95)

CLINICAL RAG PIPELINE EVALUATION SUMMARY
                         Case                                           Clinical Inquiry                                                                                           Grounded Answer Preview            Primary Source Citation
                       Case 1 What are the common symptoms and warning signs of diabe... Common Symptoms and Warning Signs of Diabetes:  - Tiredness: Carbohydrates cannot be converted into energy. - ...              diabetes.pdf (Page 1)
                       Case 2 What are the blood pressure ranges for Normal, Elevated... **Normal Blood Pressure Range:** Systolic 120-129 mm Hg, Diastolic 80-89 mm Hg  **Elevated Blood Pressure Rang...   high-blood-pressure.pdf (Page 0)
                       Case 3 What are the primary symptoms of asthma and what common... Primary Symptoms of Asthma:  - Wheezing (often occurring at night) - Periodic breathing difficulties during sl...            Pneumonia.pdf (Page 13)
       

## 11. In-Depth Discussion: Retrieval Strategy, Trade-Offs & Limitations

### 1. Retrieval Strategy Analysis

#### A. Chunking Window Trade-Offs
- **Window Size ($800$ characters)**: We selected an 800-character window based on empirical clinical parsing. Smaller chunks ($200$–$300$ characters) frequently split symptom enumerations (e.g., isolating "numbness" from "stroke"), degrading retrieval relevance. Larger chunks ($1,500$+ characters) dilute semantic density, causing irrelevant sentences to dominate the embedding vector.
- **Chunk Overlap ($150$ characters)**: Essential for preserving cross-sentence dependencies such as drug contraindications and multi-part diagnostic criteria.

#### B. Dense Vector Retrieval vs. Lexical BM25 Search
- **Dense Semantic Search (BGE)**: Excels at capturing conceptual equivalences (e.g., linking "high blood pressure" to "hypertension" or "chest discomfort" to "angina pectoris") without requiring exact keyword matches.
- **Failure Modes of Pure Dense Retrieval**: Can struggle with rare alphanumeric medical codes, exact pharmaceutical dosages (e.g., "lisinopril 10mg vs 20mg"), or acronyms that overlap with common vocabulary. In production clinical environments, a **hybrid search** combining BM25 (sparse lexical) and BGE (dense semantic) via Reciprocal Rank Fusion (RRF) provides optimal recall.

#### C. Top-$k$ Selection Dynamics ($k=3$)
- Constraining retrieval to $k=3$ chunks balances context richness against prompt saturation. With average chunk lengths of ~800 characters (~150 words), 3 chunks provide ~450–500 words of targeted clinical evidence. This fits comfortably within the LLM's high-attention receptive field, preventing the "lost-in-the-middle" phenomenon common when $k \ge 10$.

---

### 2. Architectural & Clinical Limitations

1. **Static Corpus Constraint**: The pipeline indexes 10 reference documents. Emerging clinical guidelines, new pharmacology approvals, or revised diagnostic criteria are inaccessible unless the vector store is incrementally re-indexed.
2. **Context Window vs. Table Extraction**: Highly structured dosage tables and flowcharts embedded within PDFs can suffer text-flow degradation during standard PDF text extraction. Vision-Language Document Parsers (e.g., ColPali or Nougat) could enhance tabular data preservation.
3. **Absence of Patient-Specific Multi-Modal Data**: Real-world clinical decision-making requires integrating laboratory panels, EHR vitals, imaging studies, and past medical history.
4. **Instruction Model Parameter Capacity (0.49B)**: While `Qwen2.5-0.5B-Instruct` shows remarkable instruction adherence, complex multi-step differential diagnosis and nuanced pharmacotherapy interactions benefit from scaling to larger parameter models (e.g., `Qwen2.5-1.5B` or `Qwen2.5-7B-Instruct`).

---

## ⚖️ Ethical, Regulatory & Clinical Safety Disclaimer

> **IMPORTANT CLINICAL NOTICE:**  
> This software, code, and accompanying notebook are designed strictly for **academic research, technical feasibility demonstration, and educational purposes**.  
> - This application is **NOT** a certified Software as a Medical Device (SaMD) under FDA 21 CFR Part 820, EU Medical Device Regulation (EU MDR 2017/745), or CE mark standards.  
> - It is **NOT** intended to diagnose, treat, cure, mitigate, or prevent any human disease, nor should it ever guide clinical patient management.  
> - All healthcare inquiries must be directed to licensed medical practitioners.

---

